In [0]:
from pyspark.sql.types import _parse_datatype_string

# ── CONFIGURATION ──────────────────────────────────────────────
SOURCE_PATH     = "/Volumes/dw_lnd_dev/dw_lnd/s3_ext_volume/Source_Files/"

CHECKPOINT_PATH = "/Volumes/dw_lnd_dev/dw_lnd/s3_ext_volume/_checkpoints/autoloader_stream/"

TARGET_TABLE    = "dw_lnd_dev.dw_lnd.orders_bronze"

schema_str = """
struct<
order_id: STRING,
order_date: DATE,
ship_date: DATE,
ship_mode: STRING,
customer_id: STRING,
region: STRING,
product_id: STRING,
category: STRING,
sales: DOUBLE,
quantity: INT,
profit: DOUBLE,
Job_Name: String,
RunID: String,
Ingested_Date: DATE,
SourceFileName: STRING
>
"""

schema = _parse_datatype_string(schema_str)

print("Schema is :", schema)

In [0]:

# ── AUTO LOADER ─────────────────────────────────────────────────
from pyspark.sql import functions as F

query = (
    spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("cloudFiles.useManagedFileEvents", "true")

        # Explicit schema — no inference
        .schema(schema)

        # Only pick up CSV files — ignores checkpoint folders
        .option("pathGlobFilter", "*.csv")

        # CSV options
        .option("header", "true")
        .option("delimiter", ",")
        .option("encoding", "UTF-8")
        .option("dateFormat", "dd-MM-yyyy")

        # Rescue unparseable data instead of failing
        .option("cloudFiles.rescuedDataColumn", "_rescued_data")

        .load(SOURCE_PATH)
        .na.drop("all")

        # Audit columns
        .withColumn("SourceFileName", F.col("_metadata.file_path"))
        .withColumn("Ingested_Date",  F.current_timestamp())
        .withColumn("RunID",  F.lit(dbutils.widgets.get('RunID')) )
        .withColumn("Job_Name", F.lit(dbutils.widgets.get('Job_Name') ))
        # .withColumn("RunID",  F.lit(1111))
        # .withColumn("Job_Name", F.lit("aaaa") )

        .writeStream
        .format("delta")
        .outputMode("append")
        .option("checkpointLocation", CHECKPOINT_PATH)
        .option("mergeSchema", "true")
        .trigger(availableNow=True)
        .toTable(TARGET_TABLE)
)

query.awaitTermination()
print(" Batch complete.")


In [0]:
%sql
-- drop table dw_lnd_dev.dw_lnd.orders_bronze
select  SourceFileName, Ingested_Date,
count(*) from dw_lnd_dev.dw_lnd.orders_bronze group by 1 , 2
